In [9]:
from pathlib import Path
import pandas as pd

In [10]:
archivos = sorted(Path("../data/probabilidadestadistica").glob("calificaciones*.xlsx"))
dfs = {a.name: pd.read_excel(a, header=6) for a in archivos}

In [11]:
for nombre, df in dfs.items():
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    filas_nulas = df.isna().all(axis=1)
    sin_nombre = df[["apellido", "nombre"]].isna().any(axis=1) & ~filas_nulas
    sin_usuario = df["usuario"].isna() & ~filas_nulas
    print(nombre)
    print(f"  Filas totalmente nulas: {filas_nulas.sum()}")
    print(f"  Filas sin nombre/apellido: {sin_nombre.sum()}")
    print(f"  Filas sin usuario: {sin_usuario.sum()}")

    dfs[nombre] = df.dropna(how="all").dropna(axis=1, how="all").dropna(subset=["usuario", "apellido", "nombre"])

calificaciones_alumnos_20251024_115041.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115045.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115049.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115055.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115059.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115102.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115105.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_115127.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usu

In [12]:
NOTA_MAX = 100

for nombre, df in dfs.items():
    print(f"{nombre}: notas nulas = {df['nota'].isna().sum()}")
    df["nota"] = df["nota"].fillna(f"0/{NOTA_MAX}")
    df[["nota_obtenida", "nota_max"]] = df["nota"].str.split("/", expand=True).astype(float)

calificaciones_alumnos_20251024_115041.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115045.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115049.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115055.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115059.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115102.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115105.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115127.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_115130.xlsx: notas nulas = 0


In [13]:
for nombre, df in dfs.items():
    duplicadas = df.duplicated().sum()
    usuarios_repetidos = df["usuario"].duplicated().sum()
    print(f"{nombre}: filas duplicadas = {duplicadas}, usuarios repetidos = {usuarios_repetidos}")

    dfs[nombre] = (df.drop_duplicates()
                   .sort_values("nota_obtenida", ascending=False)
                   .drop_duplicates(subset="usuario")
                   .sort_index())

df_calificaciones = pd.concat(dfs, names=["archivo"]).reset_index(level="archivo").reset_index(drop=True)
df_calificaciones = df_calificaciones.drop(columns=["nota"], errors="ignore")

df_calificaciones.to_parquet("../data/probabilidadestadistica/calificaciones_probabilidadestadistica.parquet", index=False)
df_calificaciones

calificaciones_alumnos_20251024_115041.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115045.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115049.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115055.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115059.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115102.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115105.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115127.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_115130.xlsx: filas duplicadas = 0, usuarios repetidos = 0


,archivo,usuario,apellido,nombre,observaciones,nota_obtenida,nota_max
0,calificaciones_alumnos_20251024_115041.xlsx,ALU_1001,Gómez,Manuel,aprobada,65.00,100.0
1,calificaciones_alumnos_20251024_115041.xlsx,ALU_1002,Silva,Diego,aprobada,70.00,100.0
2,calificaciones_alumnos_20251024_115041.xlsx,ALU_1004,Flores,Gabriel,aprobada,95.00,100.0
3,calificaciones_alumnos_20251024_115041.xlsx,ALU_1005,Vega,Agustín,aprobada,65.00,100.0
4,calificaciones_alumnos_20251024_115041.xlsx,ALU_1006,Ramos,Patricio,aprobada,90.00,100.0
...,...,...,...,...,...,...,...
890,calificaciones_alumnos_20251024_115130.xlsx,ALU_1179,Alvarez,Victoria,aprobada,100.00,100.0
891,calificaciones_alumnos_20251024_115130.xlsx,ALU_1224,López,Iara,aprobada,83.33,100.0
892,calificaciones_alumnos_20251024_115130.xlsx,ALU_1238,Vargas,Damián,aprobada,91.67,100.0
893,calificaciones_alumnos_20251024_115130.xlsx,ALU_1277,Silva,Gisela,aprobada,100.00,100.0


In [14]:
def a_snake_case(columnas):
    # Minúsculas, sin tildes y con "_" en lugar de espacios u otros símbolos
    return (columnas.str.strip().str.lower()
            .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
            .str.replace(r"\W+", "_", regex=True))


def limpiar_reporte(df):
    df.columns = a_snake_case(df.columns)
    df = df.dropna(how="all").dropna(axis=1, how="all").dropna(subset=["usuario", "apellido", "nombre"])
    return df[~df["usuario"].str.startswith(("STAFF", "DOC"))].reset_index(drop=True)

In [15]:
archivo = next(Path("../data/probabilidadestadistica").glob("reporte_participacion*.xlsx"))
df_participacion = pd.read_excel(archivo, header=[5, 6])
# Encabezado de dos niveles (ej. "Encuestas" / "Cantidad"): se unen en un solo nombre
df_participacion.columns = pd.Index(
    [" ".join(n for n in col if not n.startswith("Unnamed")) for col in df_participacion.columns]
)
df_participacion = limpiar_reporte(df_participacion)

df_participacion.to_parquet("../data/probabilidadestadistica/participacion_probabilidadestadistica.parquet", index=False)
df_participacion

,usuario,apellido,nombre,aula,perfil,fecha_alta,avance,interacciones,primera_interaccion,ultima_interaccion,...,wikis_cargados,wikis_paginas_con_revisiones,foros_cantidad_temas_de_debate,foros_cargados,foros_temas_de_debate_con_intervencion,foros_cantidad_de_intervenciones_del_usuario,anuncios_cantidad,anuncios_cargados,noticias_cantidad,noticias_cargados
0,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,80,1071,06/08/2025 18:04:07,23/10/2025 21:49:01,...,0,0,0,0,0,0,0,0,11,0
1,ALU_1002,Silva,Diego,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,85,1067,05/08/2025 13:21:48,22/10/2025 21:47:58,...,0,0,0,0,0,0,0,0,11,0
2,ALU_1003,Castro,Federico,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,1,11,05/08/2025 17:40:21,06/08/2025 20:02:40,...,0,0,0,0,0,0,0,0,11,0
3,ALU_1004,Flores,Gabriel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,94,1080,05/08/2025 12:01:33,23/10/2025 09:38:54,...,0,0,0,0,0,0,0,0,11,0
4,ALU_1005,Vega,Agustín,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,92,640,05/08/2025 11:55:16,22/10/2025 15:30:28,...,0,0,0,0,0,0,0,0,11,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,ALU_1315,Flores,Felipe,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,84,394,05/08/2025 22:46:40,21/10/2025 20:18:18,...,0,0,0,0,0,0,0,0,11,0
315,ALU_1316,Sosa,Emilia,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,20,220,06/08/2025 19:24:39,19/10/2025 22:33:12,...,0,0,0,0,0,0,0,0,11,0
316,ALU_1317,Morales,Nahuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,0,0,NaN,NaN,...,0,0,0,0,0,0,0,0,11,0
317,ALU_1318,Villalba,Candelaria,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,05/08/2025,24,53,06/08/2025 14:40:44,14/10/2025 12:26:08,...,0,0,0,0,0,0,0,0,11,0


In [ ]:
archivo = next(Path("../data/probabilidadestadistica").glob("reporte_trazabilidad*.xlsx"))
df_trazabilidad = limpiar_reporte(pd.read_excel(archivo, header=5))

df_trazabilidad.to_parquet("../data/probabilidadestadistica/trazabilidad_probabilidadestadistica.parquet", index=False)
df_trazabilidad
Hola

,fecha,ip_usuario,usuario,apellido,nombre,aula,perfil,seccion,accion,descripcion
0,2025-08-06 18:04:07,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Inicio,Acceder,Acceso a la sección 'Inicio'
1,2025-08-06 18:04:30,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Noticias,Ver noticia,Visualización de la noticia '&#128221; Encuest...
2,2025-08-06 18:04:40,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Noticias,Acceder,Acceso a la sección 'Noticias'
3,2025-08-06 18:04:46,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Noticias,Ver noticia,Visualización de la noticia '&#128226; ¡Bienve...
4,2025-08-06 18:05:04,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Calendario,Acceder,Acceso a la sección 'Calendario'
...,...,...,...,...,...,...,...,...,...,...
63573,2025-10-22 20:00:14,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Programa > Material de estudio,Ver archivo,Acceso al material de estudio 'GUIA_Probabilid...
63574,2025-10-22 20:00:22,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Programa > Material de estudio,Descargar,Descarga del material de estudio 'GUIA_Probabi...
63575,2025-10-22 20:00:38,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Programa,Acceder,Acceso a la sección 'Programa'
63576,2025-10-22 20:00:45,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Probabilidad y Estadística 2025,Alumnos,Programa > Textos,Ver texto,Acceso al texto 'Link a Clases en Teams : Sema...
